# CeNN V5.1 — Qwen3.5-0.8B | Automatic V5 Baseline + Head Rescue

**Reproducible no-upload research Colab · [TinyCeNN-LM](https://github.com/vtavakkoli/TinyCeNN-LM)**

**Run with an NVIDIA GPU; no uploads, Google Drive, prior ZIP archives, or saved weights are required.** The notebook first **trains a new V5-compatible DeltaCell + Local64 checkpoint** on the current training split, then initializes five V5.1 ablations from exactly those learned weights. The initial checkpoint is saved to the Colab output directory and can be reused during the same runtime **only when its settings and model/data hashes match**.

**Qwen layer 11:** preserve native full attention for GQA group 0, approximate group 1 with a gated delta memory. The Q/K/V, QK normalization, RoPE, original output gate, full model's MLP and residual pathways are all frozen. Compare the trained, frozen baseline, router-only, correction-only, router+correction, and joint memory fine-tuning.

**Scope:** causal unpadded text prefill only (`use_cache=False`). This does *not* implement cached decoding, vision, fused recurrent kernels, or an independently validated faster model. No upload is required for setup; model weights and WikiText-2 are downloaded automatically using Hugging Face's public APIs.

**Evaluation discipline:** select on the validation split, then run one diagnostic on a disjoint, predeclared WikiText-2 **training-tail** segment (not a fresh external test set). Record GPU timings, original model revision, Git SHA, baseline checkpoint hash, full-token next-token NLL, teacher agreement and cosine. Accept only if quality and *measured* speed both pass; otherwise retain native Qwen attention.


## 0 · Setup: install and clone source repository

Choose **Runtime → Change runtime type → GPU**. Run the notebook top to bottom. This notebook clones TinyCeNN-LM and uses the maintained V5/V5.1 Python modules and tests. No manual checkpoint upload or interactive input is needed. GPU training is performed in Colab, not in this document.


In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"

In [ ]:
import io, os, sys, gc, json, hashlib, subprocess, math, random, statistics, time
from pathlib import Path
from contextlib import contextmanager, nullcontext
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
                    'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))

# Fail fast on missing repository source rather than creating an untracked fork.
V51_MODULE=REPO/'src/tinycenn_lm/cenn_head_rescue_v51.py'
V51_TESTS=REPO/'tests/test_cenn_head_rescue_v51.py'
assert V51_MODULE.is_file() and V51_TESTS.is_file(), (
    'V5.1 module/tests missing from cloned repository. Open the published main-branch notebook.')
MODULE_SHA256=hashlib.sha256(V51_MODULE.read_bytes()).hexdigest()
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3
from tinycenn_lm.cenn_head_rescue_v5 import Qwen35HeadRescueV5
from tinycenn_lm.cenn_head_rescue_v51 import CeNNQueryRescueV51,warmstart_v5_delta_core
from transformers import AutoTokenizer, AutoConfig, Qwen3_5ForCausalLM
from huggingface_hub import model_info
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert torch.cuda.is_available(), 'This full Qwen experiment requires an NVIDIA CUDA runtime.'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('Git commit',GIT_SHA,'V5.1 source SHA256',MODULE_SHA256)
print('GPU:',torch.cuda.get_device_name(0),'dtype:',DTYPE)
subprocess.run([sys.executable,'-m','pytest','-q',str(V51_TESTS)],check=True)


## 1 · Settings and automatic baseline training

The experiment is **fully automatic**. `BASELINE_STEPS` fits a new V5-compatible 64-feature DeltaCell + Local64 control against the frozen native layer 11. Its checkpoint is saved under `OUT/checkpoints/` and fingerprinted against the exact model revision, source commit, training-split hash and hyperparameters. If a compatible baseline checkpoint already exists in this runtime, it is reused; otherwise baseline training starts automatically. **No previous V5 results ZIP is loaded or required.**

The V5-style baseline trained here is a *new* checkpoint, not a restoration of the separately uploaded V5 experiment. All V5.1 candidates start from the same newly calibrated weights for fair ablation.


In [ ]:
# @title No-upload experiment settings
PROFILE='quick' # @param ['quick','research']
MODEL_ID='Qwen/Qwen3.5-0.8B'
TARGET_LAYER=11
SEQ_LEN=256 if PROFILE=='quick' else 512
TRAIN_BLOCKS=48 if PROFILE=='quick' else 160
DEV_BLOCKS=24 if PROFILE=='quick' else 64
AUDIT_BLOCKS=24 if PROFILE=='quick' else 64
AUDIT_START_TOKEN=250_000
BASELINE_STEPS=140 if PROFILE=='quick' else 600
STEPS=200 if PROFILE=='quick' else 800
TASK_STEPS=15 if PROFILE=='quick' else 75
BATCH_SIZE=2
BASELINE_LR=2e-3
LR=8e-4
CHUNK_SIZE=32
SEED=3051
RANK=16
OUT=Path('/content/qwen35_cenn_head_rescue_v51')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
CANDIDATES={
 'v5_frozen':dict(router=False,correction=False,train='none'),
 'v51_router_only':dict(router=True,correction=False,train='router'),
 'v51_correction_only':dict(router=False,correction=True,train='correction'),
 'v51_router_correction':dict(router=True,correction=True,train='router_correction'),
 'v51_joint':dict(router=True,correction=True,train='joint'),
}
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Automatic V5 baseline:', BASELINE_STEPS, 'steps; V5.1:', STEPS,
      'steps per trainable candidate; no upload required')


## 2. Load the **pinned** original Qwen model
Keep the original frozen throughout. Layer 11 is Qwen3.5 **full attention**, not Gated DeltaNet. Before experiments, verify an all-exact V5 adapter reproduces native Qwen logits.

In [ ]:
REVISION=model_info(MODEL_ID).sha
print('Pinned Qwen model revision:',REVISION)
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
assert cfg.model_type=='qwen3_5_text' and cfg.num_hidden_layers==24 and cfg.hidden_size==1024
full=[i for i,t in enumerate(cfg.layer_types) if t=='full_attention']
assert TARGET_LAYER in full and full==[3,7,11,15,19,23],full
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,dtype=DTYPE,
    attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False)
model.config.use_cache=False
LAYERS=model.model.layers
NATIVE=LAYERS[TARGET_LAYER].self_attn
HD=int(NATIVE.head_dim)
QHEADS=int(NATIVE.q_proj.out_features//(2*HD))
KVHEADS=int(NATIVE.k_proj.out_features//HD)
assert (QHEADS,KVHEADS,HD)==(8,2,256),(QHEADS,KVHEADS,HD)
@contextmanager
def graft(adapter):
    assert LAYERS[TARGET_LAYER].self_attn is NATIVE
    LAYERS[TARGET_LAYER].self_attn=adapter
    try:yield
    finally:LAYERS[TARGET_LAYER].self_attn=NATIVE
sample=tokenizer('CeNN is a cellular associative memory model.',return_tensors='pt',
                 add_special_tokens=False).input_ids.to(DEVICE)
all_exact=Qwen35HeadRescueV5(NATIVE,None,exact_groups=(0,1)).eval()
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    reference=model(input_ids=sample,use_cache=False,logits_to_keep=0).logits.float()
    with graft(all_exact):parity=model(input_ids=sample,use_cache=False,logits_to_keep=0).logits.float()
max_difference=(reference-parity).abs().max().item()
print('Exact-native parity:',max_difference)
assert max_difference<.05 and bool((reference.argmax(-1)==parity.argmax(-1)).all())
del all_exact,parity,reference


## 3 · Calibration, validation and fixed diagnostic partition

Fit the baseline and all V5.1 candidates using the **same training prefix**. Select by WikiText-2 `validation`. The predeclared diagnostic window comes from the distant `train` tail after token 250,000; it is withheld from calibration/selection but is **not an independent external dataset**.


In [ ]:
from datasets import load_dataset

def build_windows(split,count,length,start_token=0):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    need=start_token+count*length
    tokens=[]
    for row in ds:
        text=row['text'].strip()
        if not text:continue
        tokens.extend(tokenizer.encode(text,add_special_tokens=False))
        tokens.append(tokenizer.eos_token_id)
        if len(tokens)>=need:break
    if len(tokens)<need:raise RuntimeError(f'{split}: {len(tokens)} tokens, need {need}')
    return torch.tensor(tokens[start_token:need],dtype=torch.long).view(count,length)
TRAIN=build_windows('train',TRAIN_BLOCKS,SEQ_LEN)
DEV=build_windows('validation',DEV_BLOCKS,SEQ_LEN)
# This split is not constructed before model selection. AUDIT_START_TOKEN is fixed above.
FINGERPRINTS={'train':hashlib.sha256(TRAIN.numpy().tobytes()).hexdigest(),
              'validation':hashlib.sha256(DEV.numpy().tobytes()).hexdigest()}
assert TRAIN_BLOCKS*SEQ_LEN < AUDIT_START_TOKEN
print('Train',TRAIN.shape,'Dev',DEV.shape,'Audit offset',AUDIT_START_TOKEN)

## 4. Capture **original teacher mixer** inputs and outputs once
Hook the actual full-attention module, retaining Qwen's RoPE tensors. None of this includes the previously held-out audit texts.

In [ ]:
TEACHER={}
@torch.inference_mode()
def forward(ids,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=ids.to(DEVICE),use_cache=False,logits_to_keep=keep)

def capture(name,blocks):
    cols={v:[] for v in ('x','y','cos','sin')}
    def hook(module,args,kwargs,result):
        inp=kwargs.get('hidden_states',args[0] if args else None)
        assert inp is not None
        val=result[0] if isinstance(result,tuple) else result
        cos,sin=kwargs['position_embeddings']
        cols['x'].append(inp.detach().to('cpu',dtype=torch.float16))
        cols['y'].append(val.detach().to('cpu',dtype=torch.float16))
        cols['cos'].append(cos.detach().cpu().to(torch.float32))
        cols['sin'].append(sin.detach().cpu().to(torch.float32))
    handle=NATIVE.register_forward_hook(hook,with_kwargs=True)
    try:
        for start in range(0,len(blocks),BATCH_SIZE):forward(blocks[start:start+BATCH_SIZE])
    finally:handle.remove()
    TEACHER[name]={key:torch.cat(val,0).contiguous() for key,val in cols.items()}
    print(name,{key:tuple(val.shape) for key,val in TEACHER[name].items()})
capture('train',TRAIN)
capture('dev',DEV)
torch.cuda.empty_cache()

## 5 · Auto-bootstrap a V5-compatible checkpoint (no upload)

Train the original **V5 DeltaCellV3 + Local64** core with exact Qwen attention for GQA group 0. Select the baseline state on the validation mixer metric. Save it to the Colab output directory with a settings fingerprint and SHA-256. This establishes a uniform *new* starting point for V5.1, and removes the need for the previous V5 archive entirely.


In [ ]:
def teacher_batch(split,indices):
    raw=TEACHER[split]
    return (raw['x'][indices].to(DEVICE,dtype=DTYPE),
            raw['y'][indices].to(DEVICE,dtype=DTYPE),
            (raw['cos'][indices].to(DEVICE),raw['sin'][indices].to(DEVICE)))

def local_metrics(pred,target):
    a,b=pred.float(),target.float()
    return ((a-b).square().mean()/b.square().mean().clamp_min(1e-8),
            F.cosine_similarity(a,b,dim=-1).mean())

def new_v5_baseline():
    core=CeNNDeltaCellV3(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                         mode='none',local_window=64).to(DEVICE)
    return Qwen35HeadRescueV5(NATIVE,core,exact_groups=(0,)).eval()

@torch.no_grad()
def validate_baseline(adapter):
    adapter.eval()
    error=energy=cosine_sum=0.;n=0
    for begin in range(0,DEV_BLOCKS,BATCH_SIZE):
        indices=torch.arange(begin,min(begin+BATCH_SIZE,DEV_BLOCKS))
        x,y,rope=teacher_batch('dev',indices)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
        a,b=pred.float(),y.float()
        error+=(a-b).square().sum().item()
        energy+=b.square().sum().item()
        cosine_sum+=F.cosine_similarity(a,b,dim=-1).sum().item()
        n+=a.shape[0]*a.shape[1]
    return dict(nmse=error/max(energy,1.e-8),cosine=cosine_sum/n)

BOOTSTRAP_SPEC=dict(model_id=MODEL_ID,model_revision=REVISION,source_git_sha=GIT_SHA,
    source_module_sha256=MODULE_SHA256,layer=TARGET_LAYER,profile=PROFILE,
    train_hash=FINGERPRINTS['train'],dev_hash=FINGERPRINTS['validation'],
    feature_dim=64,local_window=64,seq_len=SEQ_LEN,train_blocks=TRAIN_BLOCKS,
    dev_blocks=DEV_BLOCKS,chunk_size=CHUNK_SIZE,baseline_steps=BASELINE_STEPS,
    batch_size=BATCH_SIZE,lr=BASELINE_LR,seed=SEED)
BOOTSTRAP_FINGERPRINT=hashlib.sha256(
    json.dumps(BOOTSTRAP_SPEC,sort_keys=True).encode()).hexdigest()
BOOTSTRAP_PATH=OUT/'checkpoints'/f'v5_auto_L{TARGET_LAYER}_delta_f64_w64.pt'

if BOOTSTRAP_PATH.exists():
    candidate=torch.load(BOOTSTRAP_PATH,map_location='cpu',weights_only=True)
    if candidate.get('bootstrap_fingerprint')==BOOTSTRAP_FINGERPRINT:
        baseline=new_v5_baseline()
        baseline.core.load_state_dict(candidate['core_state'],strict=True)
        V5_CORE_STATE={k:v.detach().cpu().clone() for k,v in candidate['core_state'].items()}
        BOOTSTRAP_VALIDATION=candidate['validation']
        del baseline
        print('Reusing locally trained baseline, matching fingerprint',BOOTSTRAP_FINGERPRINT[:12])
    else:
        print('Existing baseline settings differ; training a new baseline automatically')
        V5_CORE_STATE=None
else:
    V5_CORE_STATE=None

if V5_CORE_STATE is None:
    torch.manual_seed(SEED+TARGET_LAYER*17)
    baseline=new_v5_baseline()
    baseline.core.requires_grad_(True)
    optimizer=torch.optim.AdamW(baseline.core.parameters(),lr=BASELINE_LR,
                                weight_decay=.01)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,T_max=BASELINE_STEPS,eta_min=BASELINE_LR*.10)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    sampling=torch.randint(0,TRAIN_BLOCKS,(BASELINE_STEPS,BATCH_SIZE),
        generator=torch.Generator().manual_seed(SEED+11))
    best=float('inf');best_sd=None;best_stats=None
    for step in range(1,BASELINE_STEPS+1):
        x,y,rope=teacher_batch('train',sampling[step-1])
        baseline.train();optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=baseline(x,position_embeddings=rope)
            nmse,cosine=local_metrics(pred,y)
            loss=nmse+.2*(1.-cosine)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'Nonfinite V5 baseline loss at step {step}')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(baseline.core.parameters(),1.)
        scaler.step(optimizer);scaler.update();scheduler.step()
        if step==1 or step==BASELINE_STEPS or step%max(1,BASELINE_STEPS//8)==0:
            stats=validate_baseline(baseline)
            score=stats['nmse']+.2*(1.-stats['cosine'])
            if score<best:
                best=score;best_stats=stats
                best_sd={k:v.detach().cpu().clone()
                         for k,v in baseline.core.state_dict().items()}
            print(f'V5 auto-bootstrap {step}/{BASELINE_STEPS} '
                  f'cos={stats["cosine"]:.5f}, NMSE={stats["nmse"]:.5f}',flush=True)
    assert best_sd is not None
    V5_CORE_STATE=best_sd
    BOOTSTRAP_VALIDATION=best_stats
    torch.save(dict(core_state=best_sd,validation=best_stats,
                    bootstrap_fingerprint=BOOTSTRAP_FINGERPRINT,bootstrap_spec=BOOTSTRAP_SPEC,
                    model_revision=REVISION,variant='rescue_g0_delta_f64_w64'),
               BOOTSTRAP_PATH)
    del baseline,optimizer,scheduler,scaler,best_sd
    torch.cuda.empty_cache()
V5_CHECKPOINT_SHA256=hashlib.sha256(BOOTSTRAP_PATH.read_bytes()).hexdigest()
BOOTSTRAP_VARIANT='rescue_g0_delta_f64_w64'
assert len(V5_CORE_STATE)==len(new_v5_baseline().core.state_dict())
print('Baseline checkpoint:',BOOTSTRAP_PATH,'SHA-256:',V5_CHECKPOINT_SHA256)
print('Baseline mixer fidelity:', BOOTSTRAP_VALIDATION)


## 6 · Train V5.1 candidates from the auto-bootstrap baseline

Each candidate receives the same newly trained 64-feature Delta + Local64 weights. The five controlled ablations are a frozen baseline, router-only, correction-only, router+correction, and joint fine-tuning. No file upload is necessary.


In [ ]:
def build_candidate(name):
    cfg=CANDIDATES['v51_joint' if name=='v51_joint_plus_kl' else name]
    if name=='v5_frozen':
        core=CeNNDeltaCellV3(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                             mode='none',local_window=64)
        core.load_state_dict(V5_CORE_STATE)
    else:
        core=CeNNQueryRescueV51(4,1,HD,feature_dim=64,chunk_size=CHUNK_SIZE,
                                local_window=64,correction_rank=RANK,
                                query_router=cfg['router'],query_correction=cfg['correction'])
        warmstart_v5_delta_core(core,V5_CORE_STATE)
    core=core.to(DEVICE)
    adapter=Qwen35HeadRescueV5(NATIVE,core,exact_groups=(0,)).eval()
    return adapter

with torch.inference_mode():
    x,y,rope=teacher_batch('dev',torch.tensor([0]))
    with torch.autocast('cuda',dtype=DTYPE):
        orig,_=build_candidate('v5_frozen')(x,position_embeddings=rope)
        improved,_=build_candidate('v51_router_correction')(x,position_embeddings=rope)
    init_diff=float((orig-improved).abs().max())
assert init_diff<0.01,init_diff
print('V5.1 warm-start output parity (BF16/FP16):',init_diff)
del orig,improved,x,y,rope


## 7 · Targeted training and validation of V5.1 variants

Train only the selected approximation core; Qwen's native parameters remain frozen. Baseline, router, correction and joint variants use the same calibration data, seed and optimizer-step budget for their comparable training stages.


In [ ]:
def set_trainables(adapter,variant):
    core=adapter.core
    core.requires_grad_(False)
    mode=CANDIDATES['v51_joint' if variant=='v51_joint_plus_kl' else variant]['train']
    if mode=='none':return []
    router=['local_logits','router_w','router_bias']
    correction=['corr_down','corr_up','corr_log_gain']
    if mode=='router':names=router
    elif mode=='correction':names=correction
    elif mode=='router_correction':names=router+correction
    elif mode=='joint':
        names=[name for name,_ in core.named_parameters()
               if not name.startswith(('cell_', 'diffusion_', 'forget_unused_'))]
    else:raise ValueError(mode)
    for name,parameter in core.named_parameters():
        if name in names:parameter.requires_grad_(True)
    enabled=[p for p in core.parameters() if p.requires_grad]
    assert len(enabled)>0 and not any(p.requires_grad for p in NATIVE.parameters())
    return enabled

@torch.no_grad()
def validate_local(adapter):
    adapter.eval();err=energy=cos_sum=0.;count=0
    for begin in range(0,DEV_BLOCKS,BATCH_SIZE):
        inds=torch.arange(begin,min(DEV_BLOCKS,begin+BATCH_SIZE))
        x,y,rope=teacher_batch('dev',inds)
        with torch.autocast('cuda',dtype=DTYPE):out,_=adapter(x,position_embeddings=rope)
        err+=(out.float()-y.float()).square().sum().item()
        energy+=y.float().square().sum().item()
        cos_sum+=F.cosine_similarity(out.float(),y.float(),dim=-1).sum().item()
        count+=out.shape[0]*out.shape[1]
    return dict(cosine=cos_sum/count,nmse=err/max(energy,1e-8))

def fit(variant):
    torch.manual_seed(SEED)
    adapter=build_candidate(variant)
    params=set_trainables(adapter,variant)
    if not params:
        return adapter,validate_local(adapter),{k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}
    opt=torch.optim.AdamW(params,lr=LR,weight_decay=.005)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=STEPS,eta_min=LR*.12)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    best_loss=float('inf');best_sd=None;best_report=None
    indices=torch.randint(0,TRAIN_BLOCKS,(STEPS,BATCH_SIZE),generator=torch.Generator().manual_seed(SEED))
    for step in range(1,STEPS+1):
        x,y,rope=teacher_batch('train',indices[step-1])
        adapter.train();opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
            nmse,cos=local_metrics(pred,y)
            loss=nmse+.2*(1-cos)
        if not bool(torch.isfinite(loss)):raise FloatingPointError((variant,step,float(loss)))
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params,1.)
        scaler.step(opt);scaler.update();sched.step()
        if step==1 or step==STEPS or step%max(1,STEPS//8)==0:
            result=validate_local(adapter)
            score=result['nmse']+.2*(1-result['cosine'])
            if score<best_loss:
                best_loss=score;best_report=result
                best_sd={k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()}
            print(f'{variant} {step}/{STEPS}: cosine={result["cosine"]:.5f} '
                  f'NMSE={result["nmse"]:.5f}',flush=True)
    assert best_sd is not None
    adapter.core.load_state_dict(best_sd);adapter.eval()
    return adapter,best_report,best_sd


## 8 · Full-token next-token evaluation and paired GPU timings

Teacher-reference logits and next-token targets are measured for the validation split only. Each candidate is evaluated in the frozen Qwen model on all next-token positions; paired GPU timing alternates native and student passes.


In [ ]:
@torch.inference_mode()
def measure_lm(blocks,reference=None):
    count=0;loss_total=0.;correct=0;agreement=0;top1=[]
    for begin in range(0,len(blocks),BATCH_SIZE):
        tok=blocks[begin:begin+BATCH_SIZE]
        idx=torch.arange(tok.shape[1]-1,device=DEVICE)
        with torch.autocast('cuda',dtype=DTYPE):
            logits=model(input_ids=tok.to(DEVICE),use_cache=False,
                         logits_to_keep=idx).logits.float()
        gold=tok[:,1:].to(DEVICE)
        logp=F.log_softmax(logits,dim=-1)
        loss_total-=logp.gather(-1,gold.unsqueeze(-1)).sum().item()
        pred=logits.argmax(-1)
        correct+=(pred==gold).sum().item();count+=gold.numel()
        if reference is None:top1.append(pred.detach().cpu().to(torch.int32))
        else:
            ref=reference[begin//BATCH_SIZE].to(DEVICE)
            assert ref.shape==pred.shape
            agreement+=(ref==pred).sum().item()
    result=dict(nll=loss_total/count,accuracy=correct/count,decisions=count)
    if reference is not None:result['teacher_agreement']=agreement/count
    return result,top1

@torch.inference_mode()
def paired_timing(adapter,blocks,warmups=3,repeats=12):
    sample=blocks[:min(len(blocks),BATCH_SIZE)].to(DEVICE)
    native_times=[];student_times=[]
    def once(use_student):
        with graft(adapter) if use_student else nullcontext():
            begin=torch.cuda.Event(enable_timing=True)
            end=torch.cuda.Event(enable_timing=True)
            begin.record()
            with torch.autocast('cuda',dtype=DTYPE):
                logits=model(input_ids=sample,use_cache=False,logits_to_keep=1).logits
            end.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(logits).all())
            return begin.elapsed_time(end)
    torch.cuda.synchronize()
    for turn in range(warmups+repeats):
        if turn%2: st=once(True);nt=once(False)
        else: nt=once(False);st=once(True)
        if turn>=warmups:native_times.append(nt);student_times.append(st)
    return dict(native_ms=float(np.median(native_times)),student_ms=float(np.median(student_times)),
                speedup=float(np.median(native_times)/np.median(student_times)),
                native_iqr_ms=float(np.subtract(*np.percentile(native_times,[75,25]))),
                student_iqr_ms=float(np.subtract(*np.percentile(student_times,[75,25]))),
                repeats=repeats)

NATIVE_DEV,NATIVE_DEV_TOP1=measure_lm(DEV)
print('Native validation',NATIVE_DEV)


## 9 · Compare all variants on validation

All five candidates start from the automatically calibrated V5-style baseline checkpoint, with no interactive inputs. Save every candidate checkpoint, metrics, provenance and the baseline SHA-256.


In [ ]:
REPORT=[]
RUN_SPEC=dict(model_revision=REVISION,git_sha=GIT_SHA,code_sha256=MODULE_SHA256,
              baseline_checkpoint_sha256=V5_CHECKPOINT_SHA256,baseline_fingerprint=BOOTSTRAP_FINGERPRINT,
              baseline_steps=BASELINE_STEPS,baseline_variant=BOOTSTRAP_VARIANT,
              profile=PROFILE,seed=SEED,steps=STEPS,task_steps=TASK_STEPS,seq_len=SEQ_LEN,
              audit_offset=AUDIT_START_TOKEN,lr=LR,rank=RANK,candidates=CANDIDATES)
FINGERPRINT=hashlib.sha256(json.dumps(RUN_SPEC,sort_keys=True).encode()).hexdigest()
for variant in CANDIDATES:
    print('\n==========',variant,'==========')
    adapter,local,sd=fit(variant)
    with graft(adapter): dev,_=measure_lm(DEV,reference=NATIVE_DEV_TOP1)
    timing=paired_timing(adapter,DEV,repeats=5,warmups=2)
    row=dict(variant=variant,layer=TARGET_LAYER,steps=0 if variant=='v5_frozen' else STEPS,
             local_dev_cosine=local['cosine'],local_dev_nmse=local['nmse'],
             dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
             dev_agreement=dev['teacher_agreement'],decisions=dev['decisions'],
             parameters=sum(p.numel() for p in adapter.core.parameters()),
             paired_speedup=timing['speedup'],native_ms=timing['native_ms'],
             student_ms=timing['student_ms'])
    torch.save(dict(core_state=sd,variant=variant,run_fingerprint=FINGERPRINT,
                    source_model_revision=REVISION,baseline_variant=BOOTSTRAP_VARIANT,
                    core_config=adapter.core.config),
               OUT/'checkpoints'/f'v51_L{TARGET_LAYER}_{variant}.pt')
    REPORT.append(row)
    pd.DataFrame(REPORT).to_csv(OUT/'validation_candidates.csv',index=False)
    print(json.dumps(row,indent=2),flush=True)
    del adapter,sd;gc.collect();torch.cuda.empty_cache()
print('Completed',len(REPORT),'candidates')

## 10 · Quality-gated validation selection

Require local cosine ≥0.95, teacher agreement ≥0.98 and validation ΔNLL ≤0.10. An ineligible candidate can be audited diagnostically but **cannot** be accepted as a faithful replacement.


In [ ]:
MIN_COSINE=.95
MIN_AGREEMENT=.98
MAX_DELTA_NLL=.10
ranked=pd.DataFrame(REPORT)
ranked['quality_gate']=((ranked.local_dev_cosine>=MIN_COSINE)&
                        (ranked.dev_agreement>=MIN_AGREEMENT)&
                        (ranked.dev_delta_nll<=MAX_DELTA_NLL))
ranked=ranked.sort_values(['quality_gate','dev_delta_nll','local_dev_nmse'],
                         ascending=[False,True,True]).reset_index(drop=True)
display(ranked[['variant','local_dev_cosine','local_dev_nmse','dev_delta_nll',
                'dev_agreement','paired_speedup','quality_gate']])
eligible=ranked[ranked.quality_gate]
if len(eligible):
    winner=eligible.sort_values(['paired_speedup','dev_delta_nll'],ascending=[False,True]).iloc[0]
    accepted=True
else:
    winner=ranked.iloc[0]
    accepted=False
print('Selected diagnostic:',winner.variant,'quality accepted:',accepted)
ranked.to_csv(OUT/'validation_ranked.csv',index=False)

## 11 · Optional end-to-end KL on the training prefix only

Disabled by default due to compute cost. Optional KL optimization uses the training prefix and frozen native teacher only; it does not train on the validation set or withheld train-tail audit. The separately named `v51_joint_plus_kl` can be restored correctly if selected.


In [ ]:
# @title Optional end-to-end distillation
ENABLE_TASK_KL=False # @param {type:'boolean'}
if ENABLE_TASK_KL:
    name='v51_joint'
    path=OUT/'checkpoints'/f'v51_L{TARGET_LAYER}_{name}.pt'
    payload=torch.load(path,map_location='cpu',weights_only=True)
    assert payload['run_fingerprint']==FINGERPRINT
    adapter=build_candidate(name)
    adapter.core.load_state_dict(payload['core_state'])
    trainable=set_trainables(adapter,name)
    opt=torch.optim.AdamW(trainable,lr=2e-5,weight_decay=.001)
    rng=np.random.default_rng(SEED+3)
    positions=torch.arange(max(0,SEQ_LEN-33),SEQ_LEN-1,2,device=DEVICE)
    for step in range(TASK_STEPS):
        choices=torch.tensor(rng.choice(len(TRAIN),BATCH_SIZE,replace=False))
        batch=TRAIN[choices].to(DEVICE)
        with torch.no_grad(),torch.autocast('cuda',dtype=DTYPE):
            teacher_logits=model(input_ids=batch,use_cache=False,
                                 logits_to_keep=positions).logits.float()
        opt.zero_grad(set_to_none=True)
        with graft(adapter),torch.autocast('cuda',dtype=DTYPE):
            student_logits=model(input_ids=batch,use_cache=False,
                                 logits_to_keep=positions).logits.float()
        tlog=F.log_softmax(teacher_logits,dim=-1).detach()
        slog=F.log_softmax(student_logits,dim=-1)
        loss=F.kl_div(slog,tlog.exp(),reduction='none').sum(-1).mean()
        assert bool(torch.isfinite(loss))
        loss.backward();torch.nn.utils.clip_grad_norm_(trainable,1.)
        opt.step()
        if (step+1)%max(1,TASK_STEPS//4)==0:print('task KL',step+1,float(loss))
    adapter.eval()
    local=validate_local(adapter)
    with graft(adapter):dev,_=measure_lm(DEV,reference=NATIVE_DEV_TOP1)
    timing=paired_timing(adapter,DEV,repeats=5,warmups=2)
    new_row=dict(variant='v51_joint_plus_kl',layer=TARGET_LAYER,steps=STEPS,
                 local_dev_cosine=local['cosine'],local_dev_nmse=local['nmse'],
                 dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
                 dev_agreement=dev['teacher_agreement'],decisions=dev['decisions'],
                 parameters=sum(p.numel() for p in adapter.core.parameters()),
                 paired_speedup=timing['speedup'],native_ms=timing['native_ms'],
                 student_ms=timing['student_ms'])
    REPORT.append(new_row)
    torch.save(dict(core_state={k:v.detach().cpu().clone() for k,v in adapter.core.state_dict().items()},
                    variant='v51_joint_plus_kl',run_fingerprint=FINGERPRINT,
                    source_model_revision=REVISION,baseline_variant=BOOTSTRAP_VARIANT,
                    core_config=adapter.core.config),
               OUT/'checkpoints'/f'v51_L{TARGET_LAYER}_v51_joint_plus_kl.pt')
    ranked=pd.DataFrame(REPORT)
    ranked['quality_gate']=((ranked.local_dev_cosine>=MIN_COSINE)&
                            (ranked.dev_agreement>=MIN_AGREEMENT)&
                            (ranked.dev_delta_nll<=MAX_DELTA_NLL))
    ranked=ranked.sort_values(['quality_gate','dev_delta_nll','local_dev_nmse'],
                              ascending=[False,True,True]).reset_index(drop=True)
    passing=ranked[ranked.quality_gate]
    if len(passing):
        winner=passing.sort_values(['paired_speedup','dev_delta_nll'],ascending=[False,True]).iloc[0]
        accepted=True
    else:
        winner=ranked.iloc[0];accepted=False
    ranked.to_csv(OUT/'validation_ranked.csv',index=False)
    print('Updated selected candidate',winner.variant,'accepted?',accepted)
else:
    print('End-to-end KL skipped; validation selection remains unchanged.')

## 12 · One withheld train-tail diagnostic and GPU audit

Only after model selection is fixed, evaluate the disjoint, predeclared training-tail tokens. This is **not** a fresh external blind test set. Compare native Qwen and the selected CeNN model at identical batch, sequence and dtype; quality and >1.05× paired speedup are both required to pass.


In [ ]:
SELECTED=str(winner.variant)
checkpoint=OUT/'checkpoints'/f'v51_L{TARGET_LAYER}_{SELECTED}.pt'
payload=torch.load(checkpoint,map_location='cpu',weights_only=True)
assert payload['run_fingerprint']==FINGERPRINT
# Restore the actual trained variant and its router/correction flags.
student=build_candidate(SELECTED)
student.core.load_state_dict(payload['core_state']);student.eval()
AUDIT=build_windows('train',AUDIT_BLOCKS,SEQ_LEN,start_token=AUDIT_START_TOKEN)
FINGERPRINTS['audit_train_tail']=hashlib.sha256(AUDIT.numpy().tobytes()).hexdigest()
assert len(set(FINGERPRINTS.values()))==3
NATIVE_AUDIT,NATIVE_AUDIT_TOP1=measure_lm(AUDIT)
with graft(student):STUDENT_AUDIT,_=measure_lm(AUDIT,reference=NATIVE_AUDIT_TOP1)
PAIRED_AUDIT=paired_timing(student,AUDIT,repeats=20,warmups=5)
DELTA_NLL=STUDENT_AUDIT['nll']-NATIVE_AUDIT['nll']
FINAL_PASS=bool(accepted and DELTA_NLL<=MAX_DELTA_NLL and
                STUDENT_AUDIT['teacher_agreement']>=MIN_AGREEMENT and
                PAIRED_AUDIT['speedup']>1.05)
FINAL=dict(architecture='CeNN V5.1 automatic baseline + query-rescue',
           model=MODEL_ID,model_revision=REVISION,repo_sha=GIT_SHA,
           baseline_origin='trained_automatically_in_this_notebook_no_upload',
           baseline_variant=BOOTSTRAP_VARIANT,baseline_steps=BASELINE_STEPS,
           baseline_fingerprint=BOOTSTRAP_FINGERPRINT,
           baseline_checkpoint_sha256=V5_CHECKPOINT_SHA256,code_sha256=MODULE_SHA256,
           profile=PROFILE,run_fingerprint=FINGERPRINT,seed=SEED,
           selected=SELECTED,validation_accepted=accepted,final_pass=FINAL_PASS,
           acceptance=dict(min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
                           max_delta_nll=MAX_DELTA_NLL,min_speedup=1.05),
           validation=ranked.to_dict('records'),
           native_audit=NATIVE_AUDIT,student_audit=STUDENT_AUDIT,
           audit_delta_nll=DELTA_NLL,paired_audit=PAIRED_AUDIT,
           fingerprints=FINGERPRINTS,
           provenance='Novel audit = disjoint WikiText-2 train tail, offset 250000 tokens; '
                      'NOT the previously consulted WikiText-2 test split',
           limitations='Full-prefix CUDA prefill only; no Qwen hybrid KV cache, variable padding, '
                       'multimodal model, fused recurrent kernel or cross-model replication.')
(OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))
print(json.dumps({key:FINAL[key] for key in
    ['selected','validation_accepted','final_pass','native_audit','student_audit',
     'audit_delta_nll','paired_audit']},indent=2))
if not FINAL_PASS:print('REJECT: retain native Qwen full attention. No speed/quality claim.')

## 13 · Optional context-length benchmarking

Diagnostic full-prefix prefill only; cached autoregressive generation is not implemented. Runtime gains must be repeated across lengths and devices before publication claims.


In [ ]:
# @title Optional length scaling
RUN_LENGTH_SWEEP=False # @param {type:'boolean'}
SWEEP=[]
if RUN_LENGTH_SWEEP:
    for length in [128,256,512,1024,2048]:
        try:
            sample=build_windows('validation',4,length)
            timing=paired_timing(student,sample,repeats=10,warmups=4)
            SWEEP.append(dict(length=length,**timing))
            print('Length',length,timing)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache();print('OOM at',length,'stopping');break
    pd.DataFrame(SWEEP).to_csv(OUT/'context_length_timing.csv',index=False)
    FINAL['length_sweep']=SWEEP
    (OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))

## 14 · Export all results automatically

A complete ZIP of V5 auto-bootstrap and all V5.1 candidates, audit JSON, metrics and plot is generated at the end. **No upload is required**; the ZIP is only a downloadable output.


In [ ]:
import shutil
frame=pd.DataFrame(REPORT)
fig,ax=plt.subplots(figsize=(11,4))
ax.barh(frame['variant'],frame['local_dev_cosine'])
ax.axvline(MIN_COSINE,ls='--',color='gray',label='cosine gate 0.95')
ax.set(xlabel='Validation teacher-mixer cosine',xlim=(0,1),
       title='Qwen3.5 CeNN V5.1 — auto-bootstrapped selective head rescue')
ax.legend();fig.tight_layout();fig.savefig(OUT/'cosine_comparison.png',dpi=160)
(OUT/'manifest.json').write_text(json.dumps(dict(
    model=MODEL_ID,revision=REVISION,git_sha=GIT_SHA,
    notebook='Qwen35_08B_CeNN_HeadRescue_V51_Warmstart_Colab.ipynb',
    profile=PROFILE,baseline_origin='automatic_training_no_upload',baseline_variant=BOOTSTRAP_VARIANT,
    baseline_steps=BASELINE_STEPS,baseline_fingerprint=BOOTSTRAP_FINGERPRINT,
    baseline_checkpoint_sha256=V5_CHECKPOINT_SHA256,source_code_sha256=MODULE_SHA256,
    candidates=list(CANDIDATES),audit_split='WikiText-2 train tail after 250000 tokens',
    quality_acceptance=accepted,final_pass=FINAL_PASS),indent=2))
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Result archive:',archive)
from IPython.display import display,FileLink
display(FileLink(archive))

### Interpretation and limitations

- **No uploads:** a compatible V5-style baseline is calibrated automatically using the frozen native model, then saved in `OUT/checkpoints/` and used to initialize all V5.1 variants.
- This baseline is **newly trained**, not the checkpoint from the prior uploaded V5 run; different weights and results are expected.
- Compare router-only, correction-only and joint adaptation to the same newly trained frozen control, with the unchanged pretrained Qwen attention group 0.
- Select using WikiText-2 validation. The withheld WikiText-2 **training-tail** diagnostic is not an independent third-party benchmark.
- If fidelity fails (cosine <0.95 or teacher agreement <0.98), or paired prefill speedup is ≤1.05, retain native Qwen full attention.
- No Qwen hybrid cache support, vision, packed/padded sequences, fused CUDA recurrence, or validated production checkpoint. Repeat with additional seeds, datasets and cached decoding before claims of speed or novelty.

All performance measurements remain **unverified until an NVIDIA GPU actually runs the notebook**.
